## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [25]:
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Load the Nasdaq-100 company information
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")

# Load the stock price change data
nasdaq = pd.read_csv("nasdaq100_price_change.csv")

# Display the first five rows of each DataFrame
print(nasdaq100_ca.head())
print(nasdaq.head())


  symbol               name  ...      cik     founded
0   AAPL         Apple Inc.  ...   320193  1976-04-01
1   ABNB             Airbnb  ...  1559720  2008-08-01
2   ADBE         Adobe Inc.  ...   796343  1982-12-01
3   ADSK           Autodesk  ...   769397  1982-01-30
4   AMAT  Applied Materials  ...     6951  1967-11-10

[5 rows x 6 columns]
  symbol      1D       5D  ...         5Y         10Y           max
0   AAPL -1.7254 -8.30086  ...  245.42031   976.99441  139245.53954
1   ABNB  2.1617 -2.21919  ...   -1.04347    -1.04347      -1.04347
2   ADBE  0.5409 -1.77817  ...  109.05718  1024.69214  251030.66399
3    ADI  0.9291 -4.03352  ...   92.81874   286.77518   26012.63736
4    ADP  2.0589  2.35462  ...   81.87224   248.40950   27613.11042

[5 rows x 12 columns]


In [26]:
# Add the year-to-date (YTD) stock performance
nasdaq100_ca = nasdaq100_ca.merge(
    nasdaq[["symbol", "ytd"]],
    on="symbol",
    how="left"
)

# Display the updated DataFrame
print(nasdaq100_ca.head())


  symbol               name        headQuarter  ...      cik     founded       ytd
0   AAPL         Apple Inc.      Cupertino, CA  ...   320193  1976-04-01  42.99992
1   ABNB             Airbnb  San Francisco, CA  ...  1559720  2008-08-01  68.66902
2   ADBE         Adobe Inc.       San Jose, CA  ...   796343  1982-12-01  57.22723
3   ADSK           Autodesk     San Rafael, CA  ...   769397  1982-01-30  10.02701
4   AMAT  Applied Materials    Santa Clara, CA  ...     6951  1967-11-10  55.46366

[5 rows x 7 columns]


In [27]:
# Function to classify a company into a sector
def classify_sector(company_name):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """Classify the company into exactly one
                of these sectors:
                Technology, Consumer Cyclical, Industrials,
                Utilities, Healthcare, Communication, Energy,
                Consumer Defensive, Real Estate, Financial.

                Return only the sector name."""
            },
            {
                "role": "user",
                "content": f"Classify this company: {company_name}"
            }
        ]
    )

    return response.choices[0].message.content.strip()


In [28]:
# Test the function with Apple
print(classify_sector("Apple Inc."))


Technology


In [29]:
# Classify each company using the OpenAI API
nasdaq100_ca["sector"] = nasdaq100_ca["name"].apply(classify_sector)

# Create the nasdaq DataFrame
nasdaq = nasdaq100_ca.copy()

# Display the first 10 companies and their sectors
print(nasdaq[["symbol", "name", "ytd", "sector"]].head(10))


  symbol                         name       ytd             sector
0   AAPL                   Apple Inc.  42.99992         Technology
1   ABNB                       Airbnb  68.66902  Consumer Cyclical
2   ADBE                   Adobe Inc.  57.22723         Technology
3   ADSK                     Autodesk  10.02701         Technology
4   AMAT            Applied Materials  55.46366         Technology
5    AMD  Advanced Micro Devices Inc.  82.45861         Technology
6   AMGN                        Amgen  -3.32887         Healthcare
7   ATVI          Activision Blizzard  19.10770         Technology
8   AVGO                Broadcom Inc.  62.07632         Technology
9   CDNS       Cadence Design Systems  45.60261         Technology


In [30]:
# Calculate the average YTD performance for each sector
sector_performance = nasdaq.groupby("sector")["ytd"].mean()

# Sort sectors from highest to lowest performance
sector_performance = sector_performance.sort_values(ascending=False)

# Display the results
print(sector_performance)


sector
Communication        49.435500
Technology           46.221933
Consumer Cyclical    18.000380
Healthcare            1.240908
Energy              -46.527970
Name: ytd, dtype: float64


In [31]:
# Select the three best-performing sectors
top_sectors = sector_performance.head(3)

print(top_sectors)


sector
Communication        49.435500
Technology           46.221933
Consumer Cyclical    18.000380
Name: ytd, dtype: float64


In [32]:
# Get the names of the top three sectors
top_sector_names = top_sectors.index.tolist()

# Select companies belonging to these sectors
top_companies = nasdaq[
    nasdaq["sector"].isin(top_sector_names)
]

# Sort companies by YTD performance
top_companies = top_companies.sort_values(
    by="ytd", ascending=False
)

# Display the companies
print(top_companies[["symbol", "name", "sector", "ytd"]])


   symbol                         name             sector        ytd
29   NVDA                       Nvidia         Technology  217.26511
26   META               Meta Platforms         Technology  153.77585
5     AMD  Advanced Micro Devices Inc.         Technology   82.45861
25   LRCX                 Lam Research         Technology   70.25344
1    ABNB                       Airbnb  Consumer Cyclical   68.66902
8    AVGO                Broadcom Inc.         Technology   62.07632
2    ADBE                   Adobe Inc.         Technology   57.22723
30   PANW           Palo Alto Networks         Technology   55.46407
4    AMAT            Applied Materials         Technology   55.46366
28   NFLX                      Netflix      Communication   49.43550
18  GOOGL      Alphabet Inc. (Class A)         Technology   47.58752
17   GOOG      Alphabet Inc. (Class C)         Technology   47.09030
9    CDNS       Cadence Design Systems         Technology   45.60261
0    AAPL                   Apple 

In [33]:
# Convert the selected stock data into text
stock_data = top_companies[
    ["symbol", "name", "sector", "ytd"]
].to_string(index=False)

# Ask OpenAI for recommendations
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """You are a stock market analyst.

            Use only the supplied stock data.

            Recommend companies from at least
            THREE different sectors.

            For each sector, recommend at least
            TWO different companies when available.

            Include each company's name, symbol,
            YTD performance, and a brief explanation.

            Do not invent company information.
            If a sector has fewer than two companies,
            clearly state that limitation."""
        },
        {
            "role": "user",
            "content": f"""
            Analyze these three sectors and their
            average YTD performance:

            {top_sectors.to_string()}

            Companies and their performance:

            {stock_data}

            Write a sector performance summary
            covering all THREE sectors and recommend
            at least two companies per sector
            where the data allows.
            """
        }
    ]
)

# Save the response
stock_recommendations = response.choices[0].message.content

print(stock_recommendations)



### Sector Performance Summary

1. **Communication Sector**
   - **Average YTD Performance**: 49.44%
   - **Recommended Companies**:
     - **Netflix (NFLX)**: YTD Performance - 49.44%. Netflix has shown strong growth in subscriber numbers and revenue recovery, reflecting its robust content pipeline and pricing strategy, making it a solid investment in the communication sector.

2. **Technology Sector**
   - **Average YTD Performance**: 46.22%
   - **Recommended Companies**:
     - **Nvidia (NVDA)**: YTD Performance - 217.27%. Nvidia leads the market due to its dominance in GPU technology and significant investments in AI, contributing to exceptional market performance.
     - **Meta Platforms (META)**: YTD Performance - 153.78%. Meta continues to innovate in social media and virtual reality, leading to strong growth potential and investor interest.

3. **Consumer Cyclical Sector**
   - **Average YTD Performance**: 18.00%
   - **Recommended Companies**:
     - **Airbnb (ABNB)**: YTD Pe